# VanMitra-AI — Module B: Digital Fencing (Ozar Village Pilot)
### Satellite-Based Boundary Monitoring — Colab Implementation

This notebook implements the **VanMitra-AI Module B Implementation Plan** for the Ozar village pilot
(Palghar District), covering:

1. Ingestion & validation of the 147-record `Ozar_Village_Dataset.csv`
2. **Real village boundary & satellite basemap retrieval** — live geocoding + OpenStreetMap boundary
   lookup for Ozar, Palghar, with a real Esri satellite tile layer on every map (falls back to a clearly
   labelled approximate outline only if there's no internet / no OSM polygon for this specific village)
3. Boundary Acquisition Module — simulated digitisation (interactive drawing needs the Flutter app), now
   producing **irregular, realistically-shaped parcel polygons that never overlap each other or the
   simulated road network**, each placed with a spatial-index-backed no-overlap search
4. **Domestic (homestead) vs. agricultural (farm) land labelling** for every parcel, via a documented,
   editable area-based rule
5. Area reconciliation (planar/UTM-metre area check, exact by construction)
6. Resolution-feasibility tagging (reliable / marginal / unreliable)
7. Two-tier map visualisation (village outline + roads + per-parcel sub-boundaries) over a real satellite
   basemap
8. Sentinel-2 acquisition via Google Earth Engine (with a synthetic-data fallback if GEE isn't authenticated)
9. NDVI / ΔNDVI computation, thresholding & clustering
10. Siamese CNN change-detection model (contrastive + BCE loss)
11. Model evaluation (IoU, precision, recall, F1, accuracy) by resolution band
12. Likely-cause tagging, risk tiering (🟢/🟡/🔴), and the final alert dashboard — coloured by tier, styled
    by land-use type, over the real satellite basemap

**How to use this notebook:**
- Run cells top to bottom.
- In **Cell 5**, upload your real `Ozar_Village_Dataset.csv` when prompted. If you skip the upload, the
  notebook generates a synthetic 147-row dataset with the same schema so every downstream cell still runs.
- **Cell 4** needs internet access (Nominatim + OpenStreetMap + a tile server) to fetch the real boundary
  and satellite imagery — this works out of the box in a normal Colab session.
- Cells are self-contained and print/plot their own output so you can sanity-check each stage before moving on.
- Each markdown cell right above a code cell tells you **what that code cell does, what it outputs, and
  roughly how long/heavy it is** to run.


## Cell 1 — Environment Setup (installs)

**What it does:** Installs the Python packages this notebook needs that aren't preloaded in Colab:
`geemap`, `earthengine-api`, `geopandas`, `shapely`, `rasterio`, `scikit-image`, `folium`, plus
`osmnx` and `geopy` (new — used by Cell 4 to fetch Ozar's *real* boundary/coordinates from OpenStreetMap).
`torch`, `numpy`, `pandas`, `matplotlib`, `pyproj` already ship with Colab.

**Output:** pip install logs (safe to ignore warnings about dependency resolver).

**Size/time:** ~1–2 minutes on first run. Only needs to run once per Colab session.


In [ ]:
!pip install -q geemap earthengine-api geopandas shapely rasterio scikit-image folium osmnx geopy
print("Environment setup complete.")


## Cell 2 — Imports

**What it does:** Imports every library used later in the notebook in one place, so later cells don't
repeat import statements. Adds `osmnx`/`geopy` (real boundary retrieval, Cell 4) and `shapely`'s
`STRtree`/`affinity`/`ops.transform` helpers (non-overlapping irregular-polygon placement, Cell 7) on top
of the original import set.

**Output:** A single confirmation line — no heavy computation here.

**Size/time:** < 1 second.


In [ ]:
import io
import json
import math
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from shapely.geometry import Polygon, Point, LineString, MultiPolygon, mapping
from shapely.ops import unary_union, transform as shp_transform
from shapely.affinity import scale as shp_scale, translate as shp_translate, rotate as shp_rotate
from shapely.strtree import STRtree
from shapely import wkt as shapely_wkt
import geopandas as gpd
from pyproj import Transformer

from skimage.morphology import binary_opening, square
from skimage.measure import label as cc_label, regionprops

import folium

import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Real-boundary retrieval deps (Cell 4) — optional; notebook falls back gracefully if unavailable.
try:
    import osmnx as ox
    OSMNX_AVAILABLE = True
except Exception:
    OSMNX_AVAILABLE = False

try:
    from geopy.geocoders import Nominatim
    GEOPY_AVAILABLE = True
except Exception:
    GEOPY_AVAILABLE = False

print("Imports OK.  osmnx available:", OSMNX_AVAILABLE, "| geopy available:", GEOPY_AVAILABLE)
import zipfile


## Cell 3 — Constants (per Report §7 / this Plan)

**What it does:** Defines the fixed parameters used throughout the pipeline so every later stage reads
from one place: Sentinel-2 pixel size, the NDVI change threshold `θ`, the minimum cluster area `A_min`,
the Siamese decision threshold `τ`, the resolution-feasibility cut-offs (900 m² / 400 m²), the parcel
no-overlap placement parameters, the domestic-vs-farmland area threshold, and the Ozar village search
query / fallback reference centre used only if live boundary retrieval (Cell 4) is unavailable.

**Output:** Printed summary of the constants.

**Size/time:** instant.


In [ ]:
PIXEL_SIDE_M = 10.0                 # Sentinel-2 pixel resolution
PIXEL_AREA_SQM = PIXEL_SIDE_M ** 2   # 100 sqm per pixel

THETA = 0.18                        # NDVI change-candidate threshold (theta ~ 0.15-0.20)
A_MIN_PIXELS = 4                    # minimum cluster area (in pixels) to keep a candidate region
TAU = 0.5                           # Siamese CNN decision threshold on p_change

RELIABLE_MIN_SQM = 900.0            # >= 9 pixels -> reliable
MARGINAL_MIN_SQM = 400.0            # 4-9 pixels  -> marginal, < 4 pixels -> unreliable

AREA_DISCREPANCY_TOLERANCE_PCT = 15.0  # Section 5.4 tolerance before flagging under_review

# --- Requirement 3: domestic (homestead) vs. agricultural (farm) land ------------
# The CSV has no explicit land-use column, so parcels are labelled by a documented, editable area rule:
# small plots are treated as homestead/residential ("Domestic/Homestead"), larger ones as cultivated
# land ("Agricultural/Farmland"). Adjust this single constant if ground-truth suggests a different cutoff.
DOMESTIC_AREA_THRESHOLD_SQM = 1000.0

# --- Requirements 2 & 4: non-overlapping, irregularly-shaped parcel placement ----
ROAD_WIDTH_M = 6.0                  # approximate carriageway width of the simulated internal road network
PARCEL_EDGE_GAP_M = 1.0             # tolerance used when checking a candidate parcel is inside the boundary
MAX_PLACEMENT_ATTEMPTS = 250        # random-search attempts per parcel before the deterministic grid fallback

# --- Requirement 1: real village location -----------------------------------------
OZAR_SEARCH_QUERY = "Ozar, Palghar District, Maharashtra, India"

# FALLBACK reference centre only — used if live geocoding/OSM lookup (Cell 4) is unavailable in this
# session (e.g. no internet). Approximate location in the Palghar tribal belt; never presented as
# authoritative (see the boundary-source label printed by Cell 4 and the Cell 21 limitations note).
OZAR_CENTRE_LAT = 19.87
OZAR_CENTRE_LON = 72.95

print("Pixel area:", PIXEL_AREA_SQM, "sqm")
print("Theta (NDVI threshold):", THETA)
print("A_min (pixels):", A_MIN_PIXELS)
print("Tau (Siamese decision threshold):", TAU)
print("Resolution bands -> reliable >=", RELIABLE_MIN_SQM, "sqm | marginal >=", MARGINAL_MIN_SQM, "sqm | else unreliable")
print("Domestic/Homestead cutoff: <", DOMESTIC_AREA_THRESHOLD_SQM, "sqm  (>= is Agricultural/Farmland)")
print("Road width:", ROAD_WIDTH_M, "m | max placement attempts/parcel:", MAX_PLACEMENT_ATTEMPTS)


## Cell 4 — Real Village Boundary & Satellite Basemap Retrieval (Requirement 1)

**What it does:** Gets Ozar village's *real* location and, where possible, its *real* administrative
boundary polygon and land area, rather than relying only on a hard-coded approximate centre:

1. Geocodes **"Ozar, Palghar District, Maharashtra, India"** via OpenStreetMap Nominatim (`geopy`).
2. Tries to pull the actual village boundary polygon from OpenStreetMap (`osmnx`) — real survey/
   administrative data, not invented — and computes its **real geodesic area** (reprojected to the local
   UTM zone, not raw degrees) in m² and hectares.
3. Registers a **real satellite imagery basemap** (Esri World Imagery tiles) that every map in this
   notebook uses as a base layer from here on, so parcels are drawn over an actual satellite view of the
   area, not a blank canvas.
4. If any step fails (no internet in this session, OSM has no polygon for this specific revenue village,
   etc.) it falls back to a clearly-labelled *approximate* outline around the fallback centre — never
   silently substituting a fake boundary for a real one without saying so.

**Output:** Printed geocoding/OSM result, the real (or fallback) village polygon `village_boundary_wgs84`,
its area in hectares, and an inline satellite-tile preview map.

**Size/time:** a few seconds; needs internet (Nominatim + OSM Overpass + tile server). Falls back
instantly and safely if offline.


In [ ]:
REAL_BOUNDARY_SOURCE = "FALLBACK"   # becomes "OSM" if a real boundary polygon is found below
village_boundary_wgs84 = None
ozar_lat, ozar_lon = OZAR_CENTRE_LAT, OZAR_CENTRE_LON

# --- Step 1: geocode the village --------------------------------------------------
if GEOPY_AVAILABLE:
    try:
        geolocator = Nominatim(user_agent="vanmitra_ozar_module_b")
        loc = geolocator.geocode(OZAR_SEARCH_QUERY, timeout=10)
        if loc is not None:
            ozar_lat, ozar_lon = loc.latitude, loc.longitude
            print(f"Geocoded '{OZAR_SEARCH_QUERY}' -> lat={ozar_lat:.5f}, lon={ozar_lon:.5f}  (Nominatim/OSM)")
        else:
            print(f"Nominatim returned no match for '{OZAR_SEARCH_QUERY}'. Using fallback centre "
                  f"lat={ozar_lat:.5f}, lon={ozar_lon:.5f}.")
    except Exception as e:
        print(f"Geocoding unavailable ({type(e).__name__}: {e}). Using fallback centre "
              f"lat={ozar_lat:.5f}, lon={ozar_lon:.5f}.")
else:
    print("geopy not installed in this session — using fallback centre.")

OZAR_CENTRE_LAT, OZAR_CENTRE_LON = ozar_lat, ozar_lon

# --- Step 2: try to fetch the real OSM administrative/place boundary -------------
if OSMNX_AVAILABLE:
    try:
        gdf_boundary = ox.geocode_to_gdf(OZAR_SEARCH_QUERY)
        if len(gdf_boundary):
            village_boundary_wgs84 = unary_union(gdf_boundary.geometry.values)
            REAL_BOUNDARY_SOURCE = "OSM"
            print("Real OpenStreetMap boundary polygon found for Ozar village.")
    except Exception as e:
        print(f"OSM boundary lookup failed ({type(e).__name__}: {e}). Falling back to an approximate outline.")
else:
    print("osmnx not installed in this session — cannot query real OSM boundary polygons.")

# --- Step 3: fallback outline (clearly labelled, only used if Step 2 didn't work) -
if village_boundary_wgs84 is None:
    # Irregular (not rectangular) fallback outline around the geocoded/fallback centre, sized generously
    # (~250 ha) so it can comfortably hold all 147 parcels + roads without overlap in Cell 7 — explicitly
    # NOT presented as an official boundary (see the Cell 21 limitations note).
    rng_b = np.random.default_rng(11)
    n_v = 14
    angles = np.sort(rng_b.uniform(0, 2 * np.pi, n_v))
    base_radius_m = 900.0
    radii = base_radius_m * (1 + rng_b.uniform(-0.35, 0.35, n_v))
    m_per_deg_lat = 111_320.0
    m_per_deg_lon = 111_320.0 * math.cos(math.radians(OZAR_CENTRE_LAT))
    pts = []
    for a, r in zip(angles, radii):
        dx_m, dy_m = r * math.cos(a), r * math.sin(a)
        pts.append((OZAR_CENTRE_LON + dx_m / m_per_deg_lon, OZAR_CENTRE_LAT + dy_m / m_per_deg_lat))
    village_boundary_wgs84 = Polygon(pts)
    print("Using a FALLBACK approximate village outline (not an official boundary).")

# --- Real/fallback area, computed geodesically (local UTM projection, not degrees) --
_gs = gpd.GeoSeries([village_boundary_wgs84], crs="EPSG:4326")
VILLAGE_UTM_CRS = _gs.estimate_utm_crs()
village_boundary_utm = _gs.to_crs(VILLAGE_UTM_CRS).iloc[0]
VILLAGE_AREA_SQM = village_boundary_utm.area

print(f"\nBoundary source: {REAL_BOUNDARY_SOURCE}")
print(f"Village reference centre: lat={OZAR_CENTRE_LAT:.5f}, lon={OZAR_CENTRE_LON:.5f}")
print(f"Village land area: {VILLAGE_AREA_SQM:,.0f} m2  (~{VILLAGE_AREA_SQM/10_000:,.2f} ha)")

# --- Real satellite basemap preview -------------------------------------------------
ESRI_SATELLITE_TILES = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
ESRI_SATELLITE_ATTR = "Esri, Maxar, Earthstar Geographics — World Imagery"

preview = folium.Map(location=[OZAR_CENTRE_LAT, OZAR_CENTRE_LON], zoom_start=15)
folium.TileLayer(tiles=ESRI_SATELLITE_TILES, attr=ESRI_SATELLITE_ATTR, name="Esri Satellite", overlay=False).add_to(preview)
folium.GeoJson(
    mapping(village_boundary_wgs84),
    style_function=lambda x: {"color": "#ffeb3b" if REAL_BOUNDARY_SOURCE == "OSM" else "#ff5252",
                               "weight": 3, "fillOpacity": 0.05},
    name=f"Ozar boundary ({REAL_BOUNDARY_SOURCE})",
).add_to(preview)
folium.LayerControl().add_to(preview)
preview


## Cell 5 — Load `Ozar_Village_Dataset.csv` (Section 4: Landowner CSV Ingestion)

**What it does:** Prompts you to upload the real `Ozar_Village_Dataset.csv`. The real file ships
`Claimant_Name_English` / `Claimant_Name_Marathi` / `Original_Claimant_Name_Raw` (transliteration
variants) rather than a single `Claimant_Name` column, so this cell also standardises on one
`Claimant_Name` display column for every downstream popup/table/export to use.
If you click "Cancel"/skip the upload dialog, it generates a synthetic 147-row dataset with the same
schema and a realistic area distribution (100–27,000 m², mean ≈ 5,059, median 4,000) purely so the rest
of the notebook has something to run against end-to-end.

**Output:** A dataframe preview (`df.head()`) and `df.shape` — should show `(147, 9)` for the real file.

**Size/time:** instant once uploaded; upload widget waits on you.


In [ ]:
try:
    from google.colab import files
    print("Please upload Ozar_Village_Dataset.csv (or click Cancel to use a synthetic demo dataset).")
    uploaded = files.upload()
except Exception:
    uploaded = {}

if uploaded:
    fname = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[fname]))
    print(f"Loaded uploaded file: {fname}")
else:
    print("No file uploaded — generating a synthetic 147-row Ozar-style dataset for demo purposes.")
    n = 147
    rng = np.random.default_rng(42)
    # log-normal-ish areas so the distribution resembles 100-27000 sqm, median ~4000
    areas = np.clip(rng.lognormal(mean=8.3, sigma=0.7, size=n), 100, 27000).round(0)
    castes = rng.choice(["ST", "SC", "OBC", "General"], size=n, p=[0.7, 0.1, 0.1, 0.1])
    df = pd.DataFrame({
        "Sr_No": range(1, n + 1),
        "District_Case_No": [f"PLG-OZR-{2021}-{i:04d}" for i in range(1, n + 1)],
        "Claimant_Name_English": [f"Claimant_{i}" for i in range(1, n + 1)],
        "Village_Name": ["Ozar"] * n,
        "Survey_No": [f"SN-{100+i}" for i in range(1, n + 1)],
        "Area_sqm": areas,
        "Caste_Category": castes,
    })

# Standardise on a single display-name column regardless of which schema was loaded.
if "Claimant_Name" not in df.columns:
    for cand in ["Claimant_Name_English", "Claimant_Name_Marathi", "Original_Claimant_Name_Raw"]:
        if cand in df.columns:
            df["Claimant_Name"] = df[cand]
            break
    else:
        df["Claimant_Name"] = "Unknown"

print("Shape:", df.shape)
df.head()


## Cell 6 — CSV Validation (Section 4)

**What it does:** Runs the three validation checks the plan specifies on import:
- `Sr_No` and `District_Case_No` uniqueness
- `Area_sqm` parses as a positive numeric value
- `Village_Name` equals "Ozar" for this pilot batch (mismatches flagged, not silently dropped)

It then adds the `boundary_status` column, initialised to `not_drawn` for every row (Section 4's
post-import state), matching the conceptual `landowners` table schema.

**Output:** A validation report (counts of issues found, if any) plus the first rows of the resulting
`landowners`-style dataframe with its new `boundary_status` column.

**Size/time:** instant.

In [ ]:
report = {}

# Uniqueness checks
report["duplicate_Sr_No"] = int(df["Sr_No"].duplicated().sum())
report["duplicate_District_Case_No"] = int(df["District_Case_No"].duplicated().sum())

# Area_sqm numeric & positive
df["Area_sqm"] = pd.to_numeric(df["Area_sqm"], errors="coerce")
report["non_numeric_or_missing_area"] = int(df["Area_sqm"].isna().sum())
report["non_positive_area"] = int((df["Area_sqm"] <= 0).sum())

# Village_Name mismatch flag (Ozar pilot batch)
mismatches = df[df["Village_Name"].str.strip().str.lower() != "ozar"]
report["village_name_mismatches"] = int(len(mismatches))

df["landowner_id"] = df["Sr_No"]  # PK stand-in
df["boundary_status"] = "not_drawn"
df["imported_at"] = pd.Timestamp.now()

print("=== CSV Validation Report (Section 4) ===")
for k, v in report.items():
    flag = "OK" if v == 0 else "FLAGGED"
    print(f"  {k}: {v}  [{flag}]")

print()
print("Area_sqm summary:")
print(df["Area_sqm"].describe()[["min", "mean", "50%", "max"]].rename({"50%": "median"}))

landowners = df.copy()
landowners.head()

## Cell 7 — Boundary Digitisation: Irregular, Non-Overlapping Parcels + Roads + Land-Use Labelling (Requirements 2–4)

**Important note (unchanged from the original plan):** In the real system, boundary geometry comes from
an **operator manually tracing each parcel on a satellite basemap** in the Flutter app's polygon-drawing
tool (Section 5.1) — that interactive step can't happen inside a Colab notebook. This cell still
*simulates* that output (tagged `boundary_status = drawn`, never presented as a real GIS/official
boundary — Report §16 Clause 4), but with three changes over the original square-parcel demo:

1. **Realistic, irregular parcel shapes (Requirement 4).** Each of the 147 parcels gets its own organic
   polygon — randomised vertex count, angular spacing, radius jitter, and rotation — instead of a uniform
   square, while its *planar area still reconciles exactly with the CSV's declared `Area_sqm`*.
2. **Guaranteed non-overlap, including roads (Requirement 2).** A simple internal road network is
   generated first (one main road + a few branches, buffered to a realistic carriageway width). Parcels
   are then placed one at a time, largest first, using a spatial-index-backed (`STRtree`) search: every
   candidate polygon is checked against *every already-placed parcel* and against the road network before
   being accepted; if random search can't find a free spot within `MAX_PLACEMENT_ATTEMPTS`, a deterministic
   shrinking-grid scan is used as a guaranteed fallback. All of this geometry math happens in the local UTM
   metre CRS (from Cell 4) for correctness, then is reprojected back to lon/lat for storage and mapping —
   so `computed_area_sqm` is an exact planar figure, not a lon/lat approximation.
3. **Domestic vs. farm land labelling (Requirement 3).** Every parcel is tagged `Domestic/Homestead` or
   `Agricultural/Farmland` using the documented `DOMESTIC_AREA_THRESHOLD_SQM` rule from Cell 3. Both
   classes get their own colour/style on every map in this notebook (Cells 10 and 20).

If the real/fallback village boundary from Cell 4 is too small to hold all 147 parcels without crowding,
it's automatically padded (scaled up about its own centroid, same shape preserved) with a printed note —
this notebook never silently shrinks or distorts parcels to force a fit.

**Output:** A `land_parcel_boundaries`-style dataframe (`boundary_id, landowner_id, geom_wkt,
land_use_type, computed_area_sqm, placement_ok`), a `roads_df` dataframe of the simulated road
centrelines, and a printed placement-success summary.

**Size/time:** a few seconds for 147 parcels (random search + deterministic grid fallback).


In [ ]:
def classify_land_use(area_sqm):
    """Requirement 3: documented area-based domestic-vs-farmland rule (Cell 3, DOMESTIC_AREA_THRESHOLD_SQM)."""
    return "Domestic/Homestead" if area_sqm < DOMESTIC_AREA_THRESHOLD_SQM else "Agricultural/Farmland"


def generate_irregular_polygon(area_sqm, rng, min_vertices=7, max_vertices=13,
                                irregularity=0.45, spikiness=0.35):
    """Organic 'blob' polygon (metres, centred on the origin) whose shoelace area equals area_sqm
    exactly. Every parcel gets its own randomised vertex count/spacing/radius/rotation so no two
    parcels look alike or read as a generic square (Requirement 4)."""
    n = int(rng.integers(min_vertices, max_vertices + 1))
    step = 2 * np.pi / n
    raw_angles = np.cumsum(rng.uniform(step * (1 - irregularity), step * (1 + irregularity), n))
    angles = raw_angles / raw_angles[-1] * 2 * np.pi
    base_r = math.sqrt(area_sqm / math.pi)  # equal-area-circle radius, used only as a starting scale
    radii = base_r * (1 + rng.uniform(-spikiness, spikiness, n))
    pts = [(r * math.cos(a), r * math.sin(a)) for a, r in zip(angles, radii)]
    poly = Polygon(pts)
    if (not poly.is_valid) or poly.area == 0:
        poly = poly.buffer(0)
    scale_factor = math.sqrt(area_sqm / poly.area)          # rescale so the area matches exactly
    poly = shp_scale(poly, xfact=scale_factor, yfact=scale_factor, origin=(0, 0))
    poly = shp_rotate(poly, math.degrees(rng.uniform(0, 2 * np.pi)), origin=(0, 0))
    return poly


def build_road_network(boundary_utm, rng):
    """A simple internal road skeleton (one main road + a few branches), clipped to the boundary."""
    minx, miny, maxx, maxy = boundary_utm.bounds
    cx, cy = boundary_utm.centroid.x, boundary_utm.centroid.y
    diag = math.hypot(maxx - minx, maxy - miny)
    main_angle = rng.uniform(0, math.pi)
    dx, dy = math.cos(main_angle) * diag, math.sin(main_angle) * diag
    main_road = LineString([(cx - dx, cy - dy), (cx, cy), (cx + dx, cy + dy)])
    roads = [main_road]
    for _ in range(int(rng.integers(2, 4))):
        t = rng.uniform(0.2, 0.8)
        branch_pt = main_road.interpolate(t, normalized=True)
        branch_angle = main_angle + math.pi / 2 + rng.uniform(-0.3, 0.3)
        blen = diag * rng.uniform(0.25, 0.5)
        bx, by = math.cos(branch_angle) * blen, math.sin(branch_angle) * blen
        roads.append(LineString([(branch_pt.x, branch_pt.y), (branch_pt.x + bx, branch_pt.y + by)]))
    clipped = [r.intersection(boundary_utm) for r in roads]
    road_lines = []
    for r in clipped:
        if r.is_empty:
            continue
        if r.geom_type == "LineString":
            road_lines.append(r)
        elif r.geom_type == "MultiLineString":
            road_lines.extend(list(r.geoms))
    road_polys = [r.buffer(ROAD_WIDTH_M / 2) for r in road_lines]
    roads_union = unary_union(road_polys) if road_polys else None
    return road_lines, roads_union


# --- Pad the working boundary if it's too small to comfortably fit all parcels + roads -----------
required_area = df["Area_sqm"].sum() * 3.0   # generous headroom for irregular packing + road corridors
if village_boundary_utm.area < required_area:
    _scale = math.sqrt(required_area / village_boundary_utm.area)
    _centroid = village_boundary_utm.centroid
    village_boundary_utm = shp_scale(village_boundary_utm, xfact=_scale, yfact=_scale,
                                      origin=(_centroid.x, _centroid.y))
    print(f"Village boundary padded (scaled x{_scale:.2f} about its own centroid, same shape preserved) "
          f"to comfortably fit {len(df)} non-overlapping parcels.")

placement_boundary = village_boundary_utm.buffer(0)
transformer_to_wgs84 = Transformer.from_crs(VILLAGE_UTM_CRS, "EPSG:4326", always_xy=True)
to_wgs84 = lambda geom: shp_transform(transformer_to_wgs84.transform, geom)

rng = np.random.default_rng(7)
road_lines_utm, roads_union_utm = build_road_network(placement_boundary, rng)

placed_polys_utm, placed_records = [], []
minx, miny, maxx, maxy = placement_boundary.bounds


def fits(candidate, roads_union, nearby_placed):
    if roads_union is not None and candidate.intersects(roads_union) and candidate.intersection(roads_union).area > 1.0:
        return False
    if not placement_boundary.contains(candidate.buffer(-PARCEL_EDGE_GAP_M)):
        return False
    for other in nearby_placed:
        if candidate.intersects(other) and candidate.intersection(other).area > 1.0:
            return False
    return True


order = landowners.sort_values("Area_sqm", ascending=False)
n_fallback_grid = 0

for _, row in order.iterrows():
    template = generate_irregular_polygon(row["Area_sqm"], rng)
    placed_ok, candidate = False, None
    tree = STRtree(placed_polys_utm) if placed_polys_utm else None

    for _attempt in range(MAX_PLACEMENT_ATTEMPTS):
        cx, cy = rng.uniform(minx, maxx), rng.uniform(miny, maxy)
        if not placement_boundary.contains(Point(cx, cy)):
            continue
        cand = shp_translate(template, xoff=cx, yoff=cy)
        nearby = [placed_polys_utm[j] for j in tree.query(cand)] if tree is not None else []
        if fits(cand, roads_union_utm, nearby):
            candidate, placed_ok = cand, True
            break

    if not placed_ok:
        n_fallback_grid += 1
        for step in (25, 15, 8, 4):
            found = False
            for gx in np.arange(minx, maxx, step):
                for gy in np.arange(miny, maxy, step):
                    if not placement_boundary.contains(Point(gx, gy)):
                        continue
                    cand = shp_translate(template, xoff=gx, yoff=gy)
                    if fits(cand, roads_union_utm, placed_polys_utm):
                        candidate, found = cand, True
                        break
                if found:
                    break
            if found:
                placed_ok = True
                break

    if candidate is None:  # last-resort placement (only if the boundary is genuinely out of room)
        candidate = shp_translate(template, xoff=placement_boundary.centroid.x, yoff=placement_boundary.centroid.y)

    placed_polys_utm.append(candidate)
    placed_records.append({
        "landowner_id": row["landowner_id"],
        "poly_utm": candidate,
        "placed_ok": placed_ok,
        "declared_area_sqm": row["Area_sqm"],
    })

n_conflicts = sum(1 for r in placed_records if not r["placed_ok"])
print(f"Placed {len(placed_records)} / {len(order)} parcels cleanly (no overlap with other parcels or roads).")
print(f"  {n_fallback_grid} parcel(s) needed the deterministic grid-search fallback.")
print(f"  {n_conflicts} parcel(s) could not find a fully clear spot (flagged in 'placement_ok' — "
      f"try re-running with a larger REQUIRED_AREA multiplier above if this is > 0).")

# --- Build the land_parcel_boundaries dataframe (lon/lat WKT, exact UTM-metre areas) --------------
records = []
for rec in placed_records:
    poly_wgs84 = to_wgs84(rec["poly_utm"])
    records.append({
        "boundary_id": rec["landowner_id"],
        "landowner_id": rec["landowner_id"],
        "geom_wkt": poly_wgs84.wkt,
        "version": 1,
        "drawn_by": "SIMULATED_OPERATOR (demo only — see Cell 7 note)",
        "status": "drawn",
        "computed_area_sqm": rec["poly_utm"].area,
        "land_use_type": classify_land_use(rec["declared_area_sqm"]),
        "placement_ok": rec["placed_ok"],
    })
parcel_boundaries = pd.DataFrame(records)
landowners = landowners.copy()
landowners.loc[:, "boundary_status"] = "drawn"

# --- Roads -> WGS84 for mapping (Cells 10, 20) -----------------------------------------------------
road_records = [{"road_id": i, "geom_wkt": to_wgs84(line).wkt} for i, line in enumerate(road_lines_utm)]
roads_df = pd.DataFrame(road_records)

# --- Final working village outline (post-padding), used by every map from here on -----------------
ozar_outline_wgs84 = to_wgs84(placement_boundary)

print(f"\nSimulated digitisation complete: {len(parcel_boundaries)} / {len(landowners)} parcels now 'drawn'.")
print(parcel_boundaries["land_use_type"].value_counts())
parcel_boundaries.head()


## Cell 8 — Area Reconciliation (Section 5.4)

**What it does:** Compares each parcel's `computed_area_sqm` (from the shoelace/geodesic calculation in
Cell 7) against the CSV's `declared_area_sqm`, computes the percentage discrepancy, and flags any parcel
beyond the configured tolerance (15%) as `under_review` rather than silently accepting it — exactly as
Section 5.4 specifies.

```
area_discrepancy_pct = | computed_area_sqm − declared_area_sqm | / declared_area_sqm × 100
```

**Output:** A histogram of discrepancy percentages, and a printed count of how many parcels were flagged
`under_review`. (In this simulated run, discrepancies come only from the lat/lon↔metres approximation, so
they should be small and few/no parcels should be flagged — a sanity check that Cell 7's geometry
generation is behaving correctly.)

**Size/time:** instant.

In [ ]:
merged = landowners.merge(
    parcel_boundaries[["landowner_id", "computed_area_sqm", "geom_wkt", "land_use_type", "placement_ok"]],
    on="landowner_id", how="left"
)
merged = merged.rename(columns={"Area_sqm": "declared_area_sqm"})
merged["area_discrepancy_pct"] = (
    (merged["computed_area_sqm"] - merged["declared_area_sqm"]).abs() / merged["declared_area_sqm"] * 100
)
merged["boundary_status"] = np.where(
    merged["area_discrepancy_pct"] > AREA_DISCREPANCY_TOLERANCE_PCT, "under_review", "locked"
)

n_flagged = int((merged["boundary_status"] == "under_review").sum())
print(f"Parcels flagged 'under_review' (> {AREA_DISCREPANCY_TOLERANCE_PCT}% discrepancy): {n_flagged} / {len(merged)}")
print(f"Parcels auto-'locked' (within tolerance): {len(merged) - n_flagged} / {len(merged)}")

plt.figure(figsize=(6, 4))
plt.hist(merged["area_discrepancy_pct"], bins=20, color="#3a7d44", edgecolor="white")
plt.axvline(AREA_DISCREPANCY_TOLERANCE_PCT, color="red", linestyle="--", label=f"{AREA_DISCREPANCY_TOLERANCE_PCT}% tolerance")
plt.xlabel("Area discrepancy (%)")
plt.ylabel("Number of parcels")
plt.title("Section 5.4 — Area Reconciliation: declared vs computed")
plt.legend()
plt.tight_layout()
plt.show()

merged[["landowner_id", "Claimant_Name", "declared_area_sqm", "computed_area_sqm", "area_discrepancy_pct", "boundary_status"]].head()

## Cell 9 — Resolution-Feasibility Tagging (Section 8)

**What it does:** Tags every parcel `reliable | marginal | unreliable` based on its declared area
relative to the Sentinel-2 10 m pixel grid:
- `reliable` ≥ 900 m² (≥ 9 pixels)
- `marginal` 400–900 m² (4–9 pixels)
- `unreliable` < 400 m² (< 4 pixels)

This operationalises the Report's Resolution Honesty Clause down to individual parcels (Section 8), and
the resulting flag will later be shown on each alert card and the map (Cells 10 & 19) rather than hidden.

**Output:** A bar chart of parcel counts per feasibility band and a printed summary matching the plan's
Section 8 statistics table.

**Size/time:** instant.

In [ ]:
def feasibility_band(area_sqm):
    if area_sqm >= RELIABLE_MIN_SQM:
        return "reliable"
    elif area_sqm >= MARGINAL_MIN_SQM:
        return "marginal"
    else:
        return "unreliable"

merged["resolution_feasibility"] = merged["declared_area_sqm"].apply(feasibility_band)

band_counts = merged["resolution_feasibility"].value_counts().reindex(["reliable", "marginal", "unreliable"]).fillna(0).astype(int)
print("=== Section 8 — Resolution Feasibility Summary ===")
print(f"Total parcels: {len(merged)}")
for band, cnt in band_counts.items():
    pct = 100 * cnt / len(merged)
    print(f"  {band:10s}: {cnt:3d} parcels ({pct:4.1f}%)")

plt.figure(figsize=(5, 4))
colors = {"reliable": "#2e7d32", "marginal": "#f9a825", "unreliable": "#c62828"}
plt.bar(band_counts.index, band_counts.values, color=[colors[b] for b in band_counts.index])
plt.ylabel("Number of parcels")
plt.title("Section 8 — Resolution-Feasibility Bands")
plt.tight_layout()
plt.show()

## Cell 10 — Ozar Village Outline Preview (Section 5.5, Requirement 1 & 4)

**What it does:** Instead of a convex hull of the (previously square) parcels, this now previews the
**actual working village boundary** used for placement (`ozar_outline_wgs84` — real OSM polygon, or a
clearly-labelled padded fallback, from Cells 4 & 6), together with the non-overlapping irregular parcels
coloured by land-use type and the simulated road network.

**Output:** The boundary's lon/lat extent and real area, and a quick matplotlib preview before the
interactive satellite map in Cell 11.

**Size/time:** instant.


In [ ]:
print("Village boundary source:", REAL_BOUNDARY_SOURCE, "(padded if needed — see Cell 7 note)")
minx, miny, maxx, maxy = gpd.GeoSeries([ozar_outline_wgs84], crs="EPSG:4326").total_bounds
print(f"  lon: [{minx:.5f}, {maxx:.5f}]   lat: [{miny:.5f}, {maxy:.5f}]")
print(f"  Working boundary area: {village_boundary_utm.area:,.0f} m2  (~{village_boundary_utm.area/10_000:,.2f} ha)")

parcel_geoms = [shapely_wkt.loads(wkt) for wkt in merged["geom_wkt"]]
domestic_mask = (merged["land_use_type"] == "Domestic/Homestead").values

fig, ax = plt.subplots(figsize=(7, 7))
gpd.GeoSeries([ozar_outline_wgs84]).boundary.plot(ax=ax, color="black", linestyle="--", linewidth=1.5)

domestic_geoms = [g for g, d in zip(parcel_geoms, domestic_mask) if d]
farm_geoms = [g for g, d in zip(parcel_geoms, domestic_mask) if not d]
if domestic_geoms:
    gpd.GeoSeries(domestic_geoms).plot(ax=ax, color="#ffb74d", edgecolor="black", linewidth=0.4, alpha=0.85)
if farm_geoms:
    gpd.GeoSeries(farm_geoms).plot(ax=ax, color="#8bc34a", edgecolor="black", linewidth=0.4, alpha=0.85)
for line_wkt in roads_df["geom_wkt"]:
    gpd.GeoSeries([shapely_wkt.loads(line_wkt)]).plot(ax=ax, color="#616161", linewidth=2.5)

from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor="#ffb74d", edgecolor="black", label="Domestic/Homestead"),
    Patch(facecolor="#8bc34a", edgecolor="black", label="Agricultural/Farmland"),
    Patch(facecolor="#616161", edgecolor="none", label="Road"),
]
ax.legend(handles=legend_handles, loc="upper right")
ax.set_title(f"Ozar parcels (irregular, non-overlapping) + roads — boundary: {REAL_BOUNDARY_SOURCE}")
plt.tight_layout()
plt.show()


## Cell 11 — Two-Tier Interactive Map over a Real Satellite Basemap (Section 6, Requirements 1–4)

**What it does:** Builds the two-tier `folium` map matching the app's CFR map style, now over a real
**Esri World Imagery satellite layer** (toggle vs. the plain "CartoDB positron" street layer via the
layer control, top right):
- **Layer 1:** dashed Ozar village outline — the real/working boundary from Cells 4 & 6
- **Layer 2:** the simulated road network
- **Layer 3:** per-landowner non-overlapping irregular sub-boundaries, coloured by **land-use type**
  (`Domestic/Homestead` vs `Agricultural/Farmland`) and shaded by `declared_area_sqm`, with a click-popup
  showing Claimant, Survey No, Land use, Declared Area, and the resolution-feasibility flag.

**Output:** An inline interactive Folium map (pan/zoom/click/toggle layers). This is the notebook
equivalent of the Flutter app's Boundary Map Screen.

**Size/time:** instant to render; satellite tiles load from the internet the first time you interact
with the map.


In [ ]:
m = folium.Map(location=[OZAR_CENTRE_LAT, OZAR_CENTRE_LON], zoom_start=16, tiles="CartoDB positron")
folium.TileLayer(tiles=ESRI_SATELLITE_TILES, attr=ESRI_SATELLITE_ATTR, name="Satellite (Esri World Imagery)", overlay=False).add_to(m)

# Layer 1: village outline (real/padded boundary from Cells 4 & 6)
folium.GeoJson(
    mapping(ozar_outline_wgs84),
    style_function=lambda x: {"color": "black", "weight": 2, "dashArray": "6,6", "fillOpacity": 0},
    name=f"Ozar village outline ({REAL_BOUNDARY_SOURCE})",
).add_to(m)

# Layer 2: roads
roads_fg = folium.FeatureGroup(name="Roads")
for line_wkt in roads_df["geom_wkt"]:
    folium.PolyLine([(lat, lon) for lon, lat in shapely_wkt.loads(line_wkt).coords],
                     color="#424242", weight=4, opacity=0.85).add_to(roads_fg)
roads_fg.add_to(m)

# Layer 3: per-landowner shaded sub-boundaries, styled by land-use type, shading by declared area
land_use_colors = {"Domestic/Homestead": "#ff8f00", "Agricultural/Farmland": "#43a047"}
max_area = merged["declared_area_sqm"].max()
for _, row in merged.iterrows():
    poly = shapely_wkt.loads(row["geom_wkt"])
    intensity = 0.25 + 0.55 * (row["declared_area_sqm"] / max_area)
    color = land_use_colors[row["land_use_type"]]
    popup_html = (
        f"<b>{row['Claimant_Name']}</b><br>"
        f"Survey No: {row['Survey_No']}<br>"
        f"Land use: {row['land_use_type']}<br>"
        f"Declared Area: {row['declared_area_sqm']:.0f} sqm<br>"
        f"Resolution: {row['resolution_feasibility']}<br>"
        f"Boundary status: {row['boundary_status']}"
    )
    folium.GeoJson(
        mapping(poly),
        style_function=lambda x, color=color, intensity=intensity: {
            "color": color, "weight": 1.2, "fillColor": color, "fillOpacity": intensity
        },
        popup=folium.Popup(popup_html, max_width=250),
    ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m


## Cell 12 — Google Earth Engine Authentication (Section 7 pipeline entry point)

**What it does:** Attempts to authenticate & initialise Earth Engine (`geemap`/`earthengine-api`), which
is the real Sentinel-2 source per Report §7/§10. **This requires a Google account with Earth Engine
access** — running this cell in your own Colab session will prompt an OAuth login the first time.

If authentication isn't available in your environment (e.g. no EE-registered account, running offline),
the notebook sets `EE_AVAILABLE = False` and Cell 13 automatically falls back to **synthetic Sentinel-2-
like NDVI patches** so every downstream cell (NDVI, change detection, CNN, metrics) still runs end to end.
This fallback is clearly labelled wherever it's used — never presented as real satellite data.

**Output:** Either "Earth Engine initialised successfully" or a fallback notice.

**Size/time:** a few seconds; first-time auth needs a one-time browser login.

In [ ]:
EE_AVAILABLE = False
try:
    import ee
    try:
        ee.Initialize()
    except Exception:
        ee.Authenticate()
        ee.Initialize()
    EE_AVAILABLE = True
    print("Earth Engine initialised successfully — real Sentinel-2 acquisition is available (Cell 13).")
except Exception as e:
    print("Earth Engine not available in this session (", type(e).__name__, ").")
    print("Falling back to SYNTHETIC Sentinel-2-like NDVI patches for the rest of this notebook.")
    print("To use real imagery: run this cell in your own Colab session with an Earth-Engine-registered Google account.")

## Cell 13 — Per-Parcel Sentinel-2 Acquisition, Cloud Masking & Normalisation (Sections 7.1)

**What it does:** For each parcel, either:
- **(if `EE_AVAILABLE`)** queries Sentinel-2 SR imagery over the parcel's bounding box for a baseline (T₁)
  and current (T₂) date window, masks clouds using the `QA60` band, and clips bands to the parcel; or
- **(fallback)** generates a synthetic NIR/RED patch pair per parcel with a small number of pixels
  proportional to the parcel's real-world area at 10 m resolution (so smaller parcels genuinely get fewer,
  noisier pixels — reproducing the Section 8 resolution problem realistically), with ~15% of parcels
  seeded with an injected "clearing" event for later validation (Cell 17).

Every patch (real or synthetic) is then **min-max normalised per band** (Section 7.1):

```
X_norm(x,y) = (X(x,y) − X_min) / (X_max − X_min)
```

**Output:** A dictionary `parcel_patches[landowner_id] = {"T1": {...}, "T2": {...}, "label": 0/1}` and a
printed count of parcels processed, plus one example patch pair plotted.

**Size/time:** instant for the synthetic fallback (147 small arrays). Real GEE calls are rate-limited by
Google's servers — expect a few minutes for 147 parcels if `EE_AVAILABLE`.

In [ ]:
def minmax_normalise(band):
    bmin, bmax = np.nanmin(band), np.nanmax(band)
    if bmax - bmin < 1e-9:
        return np.zeros_like(band)
    return (band - bmin) / (bmax - bmin)

def synthetic_parcel_patch(area_sqm, inject_change=False, rng=None):
    """Builds a small synthetic NIR/RED patch whose pixel count reflects the parcel's real area
    at Sentinel-2's 10m resolution (Section 8), with optional injected clearing for validation."""
    rng = rng or np.random.default_rng()
    n_pixels = max(1, int(round(area_sqm / PIXEL_AREA_SQM)))
    side = max(1, int(round(math.sqrt(n_pixels))))

    base_red = rng.normal(0.08, 0.01, size=(side, side)).clip(0.01, 0.5)
    base_nir = rng.normal(0.45, 0.03, size=(side, side)).clip(0.05, 0.9)  # healthy vegetation: high NIR

    t1_red, t1_nir = base_red.copy(), base_nir.copy()
    t2_red, t2_nir = base_red.copy(), base_nir.copy()

    label = 0
    if inject_change:
        label = 1
        # simulate clearing: NIR drops, RED rises over a sub-region of T2
        r0, r1 = 0, max(1, side)
        t2_nir[r0:r1] = (t2_nir[r0:r1] * 0.4).clip(0.02, 0.9)
        t2_red[r0:r1] = (t2_red[r0:r1] * 1.8).clip(0.01, 0.6)
    else:
        # mild seasonal noise only
        t2_nir = (t2_nir + rng.normal(0, 0.02, size=t2_nir.shape)).clip(0.05, 0.9)
        t2_red = (t2_red + rng.normal(0, 0.01, size=t2_red.shape)).clip(0.01, 0.5)

    return {"T1": {"NIR": t1_nir, "RED": t1_red}, "T2": {"NIR": t2_nir, "RED": t2_red}, "label": label}

rng = np.random.default_rng(123)
parcel_patches = {}

if EE_AVAILABLE:
    # Real Earth Engine acquisition path (Section 7, Report Sec 10/11).
    import ee
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    for _, row in merged.iterrows():
        try:
            poly = shapely_wkt.loads(row['geom_wkt'])
            coords = list(poly.exterior.coords)
            region = ee.Geometry.Polygon([coords])
            def get_masked(date_start, date_end):
                col = (s2.filterBounds(region)
                          .filterDate(date_start, date_end)
                          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))
                def mask_clouds(img):
                    qa = img.select('QA60')
                    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
                    return img.updateMask(mask)
                return col.map(mask_clouds).median().clip(region)
            img_t1 = get_masked('2024-01-01', '2024-03-01')
            img_t2 = get_masked('2024-06-01', '2024-08-01')
            # NOTE: pulling pixel arrays client-side needs ee->numpy (e.g. via getDownloadURL / xee);
            # left as a placeholder for the full production job (Report Sec 11).
            parcel_patches[row['landowner_id']] = {"ee_image_t1": img_t1, "ee_image_t2": img_t2, "label": None}
        except Exception as e:
            parcel_patches[row['landowner_id']] = synthetic_parcel_patch(row['declared_area_sqm'], False, rng)
    print(f"Queried Earth Engine image objects for {len(parcel_patches)} parcels (pixel extraction left as production TODO).")
else:
    inject_flags = rng.random(len(merged)) < 0.15  # ~15% of parcels get an injected clearing event
    for (idx, row), inject in zip(merged.iterrows(), inject_flags):
        parcel_patches[row["landowner_id"]] = synthetic_parcel_patch(row["declared_area_sqm"], inject, rng)
    n_changed = sum(1 for p in parcel_patches.values() if p["label"] == 1)
    print(f"Generated synthetic patch pairs for {len(parcel_patches)} parcels ({n_changed} with injected clearing).")

# Apply min-max normalisation (Section 7.1) to one example and plot it
example_id = merged["landowner_id"].iloc[0]
example = parcel_patches[example_id]
if "NIR" in example.get("T1", {}):
    t1_nir_n = minmax_normalise(example["T1"]["NIR"])
    t2_nir_n = minmax_normalise(example["T2"]["NIR"])
    fig, axes = plt.subplots(1, 2, figsize=(7, 3))
    axes[0].imshow(t1_nir_n, cmap="YlGn"); axes[0].set_title("T1 NIR (normalised)")
    axes[1].imshow(t2_nir_n, cmap="YlGn"); axes[1].set_title("T2 NIR (normalised)")
    for a in axes: a.axis("off")
    plt.tight_layout()
    plt.show()

## Cell 14 — NDVI, ΔNDVI, Thresholding & Clustering (Sections 7.2, 7.3)

**What it does:** For every parcel with pixel data (synthetic or extracted), computes:

```
NDVI(x,y) = (NIR − RED) / (NIR + RED)
ΔNDVI(x,y) = NDVI_T2(x,y) − NDVI_T1(x,y)
```

then builds the binary change-candidate mask `B(x,y) = 1 if ΔNDVI(x,y) < −θ`, applies a morphological
opening to suppress salt-and-pepper noise, extracts connected components (8-connectivity), and keeps
only components with `|C| ≥ A_min` pixels as alert candidates (Section 7.3). The retained cluster's
real-world area (`|C| × pixel_area`) becomes each parcel's "Area Affected" figure.

**Output:** A summary dataframe (`landowner_id, max_cluster_pixels, area_affected_sqm, has_candidate`)
and one example ΔNDVI heatmap + candidate mask plotted side by side.

**Size/time:** instant for 147 small parcels.

In [ ]:
def compute_ndvi(nir, red):
    return (nir - red) / (nir + red + 1e-9)

def candidate_clusters(delta_ndvi, theta=THETA, a_min=A_MIN_PIXELS):
    B = (delta_ndvi < -theta).astype(np.uint8)
    if B.size > 1:
        B_open = binary_opening(B, square(min(2, B.shape[0]))).astype(np.uint8)
    else:
        B_open = B
    labeled = cc_label(B_open, connectivity=2)
    kept_pixels = 0
    for region in regionprops(labeled):
        if region.area >= a_min:
            kept_pixels += region.area
    return kept_pixels, B_open

change_rows = []
example_delta = None
for landowner_id, patch in parcel_patches.items():
    if "NIR" not in patch.get("T1", {}):
        continue  # real-EE image objects without extracted arrays yet (production TODO, Cell 13)
    ndvi_t1 = compute_ndvi(patch["T1"]["NIR"], patch["T1"]["RED"])
    ndvi_t2 = compute_ndvi(patch["T2"]["NIR"], patch["T2"]["RED"])
    delta = ndvi_t2 - ndvi_t1
    kept_pixels, mask = candidate_clusters(delta)
    change_rows.append({
        "landowner_id": landowner_id,
        "max_cluster_pixels": kept_pixels,
        "area_affected_sqm": kept_pixels * PIXEL_AREA_SQM,
        "has_candidate": kept_pixels > 0,
        "true_label": patch.get("label"),
    })
    if landowner_id == example_id:
        example_delta, example_mask = delta, mask

change_df = pd.DataFrame(change_rows)
print(f"Parcels with at least one retained change candidate: {int(change_df['has_candidate'].sum())} / {len(change_df)}")
change_df.head()

In [ ]:
if example_delta is not None:
    fig, axes = plt.subplots(1, 2, figsize=(7, 3))
    im0 = axes[0].imshow(example_delta, cmap="RdYlGn", vmin=-1, vmax=1)
    axes[0].set_title("Delta NDVI (example parcel)")
    plt.colorbar(im0, ax=axes[0], fraction=0.046)
    axes[1].imshow(example_mask, cmap="gray")
    axes[1].set_title("Candidate change mask B'")
    for a in axes: a.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No example patch available to plot (check Cell 13 output).")

## Cell 15 — Siamese CNN Architecture (Section 7.4)

**What it does:** Defines the Siamese change-detection network per the plan's math spec:
- A small shared-weight CNN encoder `f_θ` mapping each patch to an embedding: `e1 = f_θ(P_T1)`, `e2 = f_θ(P_T2)`
- Distance/similarity functions: `d(e1,e2) = ||e1-e2||_p` (L1/L2) and cosine similarity
- A classification head: `p_change = σ(wᵀ·d(e1,e2) + b)`
- Two loss functions: **contrastive loss** (for the embedding branch) and **binary cross-entropy** (for the
  classification head), matching Section 7.4 exactly

The encoder here uses a small 2-conv-layer CNN (since patches are tiny — some parcels are only a few
pixels wide, per Section 8) rather than a deep architecture, which would overfit/degenerate on 1–4 pixel
inputs.

**Output:** A printed model summary (parameter count) — no training yet, that's Cell 15.

**Size/time:** instant; model is intentionally lightweight (<30MB target, Report §13/Section 11).

In [ ]:
class SharedEncoder(nn.Module):
    # Shared-weight encoder f_theta (Section 7.4). Uses adaptive pooling so it accepts
    # variable/tiny patch sizes (down to 1x1 pixels) without shape errors.
    def __init__(self, in_channels=2, embed_dim=32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(32, embed_dim)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x).flatten(1)
        return self.fc(x)


class SiameseChangeNet(nn.Module):
    # Full Siamese network: shared encoder + classification head (Section 7.4).
    def __init__(self, in_channels=2, embed_dim=32, distance="l2"):
        super().__init__()
        self.encoder = SharedEncoder(in_channels, embed_dim)
        self.distance = distance
        self.head = nn.Linear(1, 1)

    def embed(self, x):
        return self.encoder(x)

    def distance_fn(self, e1, e2):
        if self.distance == "l1":
            return torch.norm(e1 - e2, p=1, dim=1, keepdim=True)
        return torch.norm(e1 - e2, p=2, dim=1, keepdim=True)

    def cosine_similarity(self, e1, e2):
        return F.cosine_similarity(e1, e2, dim=1)

    def forward(self, x1, x2):
        e1, e2 = self.embed(x1), self.embed(x2)
        d = self.distance_fn(e1, e2)
        p_change = torch.sigmoid(self.head(d)).squeeze(1)
        return p_change, e1, e2, d.squeeze(1)


def contrastive_loss(e1, e2, y, margin=1.0):
    # Section 7.4: y=0 (no change) -> pull together; y=1 (change) -> push apart by >= margin.
    d = torch.norm(e1 - e2, p=2, dim=1)
    loss_no_change = (1 - y) * 0.5 * d.pow(2)
    loss_change = y * 0.5 * torch.clamp(margin - d, min=0).pow(2)
    return (loss_no_change + loss_change).mean()

bce_loss_fn = nn.BCELoss()

model = SiameseChangeNet(in_channels=2, embed_dim=32, distance="l2")
n_params = sum(p.numel() for p in model.parameters())
print(f"SiameseChangeNet initialised — {n_params:,} parameters (well under the <30MB on-device target).")

## Cell 16 — Training the Siamese CNN (contrastive + BCE, Section 7.4 / Phase 6)

**What it does:** Assembles a labelled training set of before/after patch pairs. Since Ozar-specific
labelled ground truth doesn't exist yet, this uses the **Tier 4 synthetic/reconstructed fallback**
(Section 3's data-source table) — extra synthetic clearing/no-change pairs generated the same way as
Cell 13, resized to a common patch size via padding so they can be batched. Trains for a small number of
epochs with:
- **contrastive loss** on the embedding branch
- **binary cross-entropy** on the classification head (trained jointly here, as the plan allows: "trains
  with contrastive loss for the embedding branch and fine-tunes the classification head with BCE")

**Output:** A per-epoch loss printout and a loss-curve plot. This is a lightweight demo training run, not
production-scale training — for a real deployment you'd train substantially longer on FFC-derived +
augmented data (Report §7.4, Phase 6).

**Size/time:** a few seconds on CPU (tiny model, tiny synthetic dataset, few epochs).

In [ ]:
PATCH_SIZE = 20  # common padded size for batching tiny/variable parcel patches (covers up to ~27,000 sqm parcels at 10m/px)

def pad_to_size(arr, size=PATCH_SIZE):
    out = np.zeros((size, size), dtype=np.float32)
    h, w = arr.shape
    h, w = min(h, size), min(w, size)
    out[:h, :w] = arr[:h, :w]
    return out

def make_training_set(n_samples=300, rng=None):
    rng = rng or np.random.default_rng(99)
    X1, X2, Y = [], [], []
    for _ in range(n_samples):
        area = rng.uniform(100, 27000)
        inject = rng.random() < 0.5
        patch = synthetic_parcel_patch(area, inject, rng)
        t1 = np.stack([pad_to_size(patch["T1"]["NIR"]), pad_to_size(patch["T1"]["RED"])])
        t2 = np.stack([pad_to_size(patch["T2"]["NIR"]), pad_to_size(patch["T2"]["RED"])])
        X1.append(t1); X2.append(t2); Y.append(patch["label"])
    return (torch.tensor(np.array(X1), dtype=torch.float32),
            torch.tensor(np.array(X2), dtype=torch.float32),
            torch.tensor(np.array(Y), dtype=torch.float32))

X1_train, X2_train, Y_train = make_training_set(300)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 15
loss_history = []

model.train()
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    p_change, e1, e2, d = model(X1_train, X2_train)
    l_contrastive = contrastive_loss(e1, e2, Y_train, margin=1.0)
    l_bce = bce_loss_fn(p_change.clamp(1e-6, 1 - 1e-6), Y_train)
    loss = l_contrastive + l_bce
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if epoch % 3 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch {epoch+1:2d}/{EPOCHS} | contrastive={l_contrastive.item():.4f} | bce={l_bce.item():.4f} | total={loss.item():.4f}")

plt.figure(figsize=(5, 3))
plt.plot(loss_history, marker="o")
plt.xlabel("Epoch"); plt.ylabel("Total loss")
plt.title("Siamese CNN training loss (demo run on synthetic Tier-4 data)")
plt.tight_layout()
plt.show()

## Cell 17 — Likely-Cause Tagging (Section 7.5, rule-based v1)

**What it does:** Applies the rule-based "Likely Cause" heuristic to every parcel with a retained change
candidate (from Cell 13):
- Large, compact, regular cluster with a strong ΔNDVI drop → `Illegal Clearing / Logging`
- Small, dispersed clusters with moderate drop → `Possible Seasonal Variation` (kept at 🟡, not escalated)
- Anything else confirmed → `Unclassified Change — Manual FRC Review Recommended`

This is explicitly a v1 heuristic, not a trained classifier (Section 7.5 / Section 13).

**Output:** A value-counts table of assigned cause tags among parcels with a candidate.

**Size/time:** instant.

In [ ]:
def tag_likely_cause(area_affected_sqm, mean_abs_delta):
    if area_affected_sqm >= 300 and mean_abs_delta >= THETA * 1.3:
        return "Illegal Clearing / Logging"
    elif area_affected_sqm < 300 and mean_abs_delta < THETA * 1.3:
        return "Possible Seasonal Variation"
    else:
        return "Unclassified Change — Manual FRC Review Recommended"

cause_rows = []
for landowner_id, patch in parcel_patches.items():
    if "NIR" not in patch.get("T1", {}):
        continue
    ndvi_t1 = compute_ndvi(patch["T1"]["NIR"], patch["T1"]["RED"])
    ndvi_t2 = compute_ndvi(patch["T2"]["NIR"], patch["T2"]["RED"])
    delta = ndvi_t2 - ndvi_t1
    row = change_df[change_df["landowner_id"] == landowner_id].iloc[0]
    if row["has_candidate"]:
        mean_abs_delta = float(np.abs(delta[delta < -THETA]).mean()) if (delta < -THETA).any() else 0.0
        cause_rows.append({
            "landowner_id": landowner_id,
            "area_affected_sqm": row["area_affected_sqm"],
            "likely_cause": tag_likely_cause(row["area_affected_sqm"], mean_abs_delta),
        })

cause_df = pd.DataFrame(cause_rows)
print(f"Parcels with a candidate change and an assigned Likely-Cause tag: {len(cause_df)}")
if len(cause_df):
    print(cause_df["likely_cause"].value_counts())
cause_df.head()

## Cell 18 — Model Evaluation: IoU, Precision, Recall, F1, Accuracy (Section 7.6)

**What it does:** Runs the trained Siamese CNN over the same synthetic patch pairs used in Cell 13/14
(which carry ground-truth `label` since they were synthetically injected) and computes:

```
IoU_i = |A_i ∩ B_i| / |A_i ∪ B_i|
Precision = TP / (TP + FP)
Recall    = TP / (TP + FN)
F1        = 2 · Precision · Recall / (Precision + Recall)
Accuracy  = (TP + TN) / (TP + TN + FP + FN)
```

Per Section 7.6/Section 8, these are reported **overall AND broken out per resolution-feasibility band**
(reliable/marginal/unreliable), so any accuracy degradation on the smallest parcels is visible rather than
averaged away.

**Output:** A confusion-matrix-derived metrics table, once overall and once per resolution band.

**Size/time:** instant (147 parcels, tiny model, CPU inference).

In [ ]:
def evaluate(landowner_ids, model, parcel_patches, merged_df):
    model.eval()
    TP = FP = TN = FN = 0
    ious = []
    with torch.no_grad():
        for lid in landowner_ids:
            patch = parcel_patches[lid]
            if "NIR" not in patch.get("T1", {}) or patch.get("label") is None:
                continue
            t1 = np.stack([pad_to_size(patch["T1"]["NIR"]), pad_to_size(patch["T1"]["RED"])])
            t2 = np.stack([pad_to_size(patch["T2"]["NIR"]), pad_to_size(patch["T2"]["RED"])])
            x1 = torch.tensor(t1[None], dtype=torch.float32)
            x2 = torch.tensor(t2[None], dtype=torch.float32)
            p_change, e1, e2, d = model(x1, x2)
            pred = int(p_change.item() > TAU)
            true = int(patch["label"])

            if pred == 1 and true == 1: TP += 1
            elif pred == 1 and true == 0: FP += 1
            elif pred == 0 and true == 0: TN += 1
            elif pred == 0 and true == 1: FN += 1

            row = change_df[change_df["landowner_id"] == lid]
            if len(row):
                pred_mask_area = row.iloc[0]["max_cluster_pixels"]
                true_mask_area = patch["T1"]["NIR"].size if true == 1 else 0
                inter = min(pred_mask_area, true_mask_area)
                union = max(pred_mask_area, true_mask_area, 1)
                ious.append(inter / union)

    precision = TP / (TP + FP) if (TP + FP) else 0.0
    recall = TP / (TP + FN) if (TP + FN) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) else 0.0
    mean_iou = float(np.mean(ious)) if ious else 0.0
    return {"n": TP+FP+TN+FN, "TP": TP, "FP": FP, "TN": TN, "FN": FN,
            "precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy, "mean_IoU": mean_iou}

print("=== Overall metrics (Section 7.6) ===")
overall = evaluate(merged["landowner_id"], model, parcel_patches, merged)
for k, v in overall.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

print()
print("=== Per resolution-feasibility band (Section 8) ===")
for band in ["reliable", "marginal", "unreliable"]:
    ids = merged.loc[merged["resolution_feasibility"] == band, "landowner_id"]
    if len(ids) == 0:
        continue
    metrics = evaluate(ids, model, parcel_patches, merged)
    print(f"[{band}] n={metrics['n']}  precision={metrics['precision']:.3f}  recall={metrics['recall']:.3f}  "
          f"f1={metrics['f1']:.3f}  accuracy={metrics['accuracy']:.3f}  mean_IoU={metrics['mean_IoU']:.3f}")

## Cell 19 — Per-Parcel Monitoring Orchestration & Risk Tiering (Section 7 loop, Section 12)

**What it does:** Runs the full per-parcel loop described in Section 7's diagram — for every parcel with
`boundary_status in (drawn, locked)`, it pulls the (already-computed) NDVI/ΔNDVI/candidate results, runs
the trained model's decision, applies **Section 12's risk-tiering rule**:

- 🟢 **Green** — no significant change detected
- 🟡 **Yellow** — localised change below threshold, OR the parcel is `marginal`/`unreliable` with any
  candidate signal
- 🔴 **Red** — `ΔNDVI < −θ` over the minimum cluster area, inside a `reliable`-band parcel

and stores the result **against that landowner_id only** — never merged with any other parcel (Section
7's "never aggregated" requirement).

**Output:** The final `alerts` dataframe — one row per parcel — with tier, area affected, likely cause,
and resolution-feasibility flag. This is what would be shown on the alert cards (Section 6) and dispatched
in Section 11's deployment pipeline.

**Size/time:** instant for 147 parcels.

In [ ]:
def assign_tier(row):
    if not row["has_candidate"]:
        return "green"
    if row["resolution_feasibility"] in ("marginal", "unreliable"):
        return "yellow"
    if row["area_affected_sqm"] >= A_MIN_PIXELS * PIXEL_AREA_SQM:
        return "red"
    return "yellow"

alerts = merged.merge(change_df, on="landowner_id", how="left")
alerts["has_candidate"] = alerts["has_candidate"].fillna(False)
alerts["area_affected_sqm"] = alerts["area_affected_sqm"].fillna(0.0)
alerts["tier"] = alerts.apply(assign_tier, axis=1)
alerts = alerts.merge(cause_df[["landowner_id", "likely_cause"]], on="landowner_id", how="left")
alerts["likely_cause"] = alerts["likely_cause"].fillna("No change detected")
alerts["detected_date"] = pd.Timestamp.now().normalize()

tier_counts = alerts["tier"].value_counts().reindex(["green", "yellow", "red"]).fillna(0).astype(int)
print("=== Section 12 — Risk Tiering Summary (per parcel, never aggregated) ===")
for tier, cnt in tier_counts.items():
    print(f"  {tier:6s}: {cnt} parcels")

alerts_view = alerts[[
    "landowner_id", "Claimant_Name", "Survey_No", "declared_area_sqm", "land_use_type",
    "resolution_feasibility", "area_affected_sqm", "tier", "likely_cause", "detected_date"
]].sort_values("tier")
alerts_view.head(15)

## Cell 20 — Final Two-Tier Map, Risk Tier + Land-Use, over a Real Satellite Basemap (Sections 6 & 12/19, Requirements 1–4)

**What it does:** Re-renders the two-tier Folium map from Cell 11 over the same real Esri satellite
basemap, this time colouring each parcel by its current alert tier (🟢/🟡/🔴) and outlining
`Domestic/Homestead` parcels with a dashed border (vs. a solid border for `Agricultural/Farmland`) so
land-use is visible at a glance alongside risk. Popups show the full alert-card fields: **Land use, Area
Affected, Detected date, Likely Cause, and resolution-feasibility flag** — matching Section 6's spec.

**Output:** The final at-a-glance risk map for the Ozar pilot, with roads and the real village outline.

**Size/time:** instant.


In [ ]:
tier_colors = {"green": "#43a047", "yellow": "#fdd835", "red": "#e53935"}

m2 = folium.Map(location=[OZAR_CENTRE_LAT, OZAR_CENTRE_LON], zoom_start=16, tiles="CartoDB positron")
folium.TileLayer(tiles=ESRI_SATELLITE_TILES, attr=ESRI_SATELLITE_ATTR, name="Satellite (Esri World Imagery)", overlay=False).add_to(m2)

folium.GeoJson(
    mapping(ozar_outline_wgs84),
    style_function=lambda x: {"color": "black", "weight": 2, "dashArray": "6,6", "fillOpacity": 0},
    name=f"Ozar village outline ({REAL_BOUNDARY_SOURCE})",
).add_to(m2)

roads_fg2 = folium.FeatureGroup(name="Roads")
for line_wkt in roads_df["geom_wkt"]:
    folium.PolyLine([(lat, lon) for lon, lat in shapely_wkt.loads(line_wkt).coords],
                     color="#424242", weight=4, opacity=0.85).add_to(roads_fg2)
roads_fg2.add_to(m2)

for _, row in alerts.iterrows():
    poly = shapely_wkt.loads(row["geom_wkt"])
    color = tier_colors[row["tier"]]
    style = {"color": color, "weight": 1.5, "fillColor": color, "fillOpacity": 0.6}
    if row["land_use_type"] == "Domestic/Homestead":
        style["dashArray"] = "4,3"
    popup_html = (
        f"<b>{row['Claimant_Name']}</b> (Survey {row['Survey_No']})<br>"
        f"Land use: {row['land_use_type']}<br>"
        f"Tier: {row['tier'].upper()}<br>"
        f"Area Affected: {row['area_affected_sqm']:.0f} sqm<br>"
        f"Detected date: {row['detected_date'].date()}<br>"
        f"Likely Cause: {row['likely_cause']}<br>"
        f"Resolution feasibility: {row['resolution_feasibility']}"
    )
    folium.GeoJson(
        mapping(poly),
        style_function=lambda x, style=style: style,
        popup=folium.Popup(popup_html, max_width=260),
    ).add_to(m2)

folium.LayerControl(collapsed=False).add_to(m2)
m2


## Cell 21 — Export Alerts & Summary Report (Section 11 deployment hand-off)

**What it does:** Writes the final per-parcel alerts table (now including `land_use_type`) to
`ozar_alerts.csv` (the artefact that, in production, feeds the FastAPI Geospatial Service's weekly
scheduled job and the SMS/in-app dispatch, Section 11), and prints a plain-language summary covering the
limitations this plan requires to be stated plainly (Section 13):

- boundary **geometry** here is simulated/demo digitisation (irregular, non-overlapping, road-aware), not
  from a real operator's traced boundaries
- the village **outline** is real OpenStreetMap data when available (`REAL_BOUNDARY_SOURCE == "OSM"`),
  otherwise a clearly-labelled approximate fallback — never presented as an official cadastral boundary
- **land-use labelling** (domestic vs. farmland) is a documented area-threshold rule, not a verified
  ground survey
- imagery is synthetic unless `EE_AVAILABLE` was `True`
- likely-cause tagging is a rule-based v1 heuristic, not a validated classifier

**Output:** `ozar_alerts.csv` written to the Colab filesystem (downloadable via the Files panel) and a
printed summary block.

**Size/time:** instant.


In [ ]:
alerts_view.to_csv("ozar_alerts.csv", index=False)

print("Saved: ozar_alerts.csv  (", len(alerts_view), "rows )")
print()
print("=== Run Summary ===")
print(f"Parcels processed: {len(merged)}")
print(f"Village boundary source: {REAL_BOUNDARY_SOURCE}")
print(f"Resolution bands: {dict(band_counts)}")
print(f"Risk tiers: {dict(tier_counts)}")
print(f"Land-use split: {dict(merged['land_use_type'].value_counts())}")
print(f"Parcels needing placement fallback / conflicts: "
      f"{int((~merged['placement_ok']).sum())} / {len(merged)}")
print(f"Earth Engine imagery used: {EE_AVAILABLE}")
print()
print("Limitations carried into this run (Section 13):")
print(" - Boundary geometry: SIMULATED (Cell 7) — irregular, non-overlapping, road-aware, but not a real")
print("   operator digitisation or an official GIS dataset.")
print(f" - Village outline: {REAL_BOUNDARY_SOURCE} (Cell 4) — real OpenStreetMap polygon when available,")
print("   otherwise a clearly-labelled approximate fallback; padded if needed to fit all parcels (Cell 7).")
print(" - Land-use labelling: documented area-threshold rule (Cell 3/7), not a verified ground survey.")
print(" - Imagery: " + ("REAL Sentinel-2 (Earth Engine)" if EE_AVAILABLE else "SYNTHETIC") + " — see Cell 13/14.")
print(" - Likely-cause tagging: rule-based v1 heuristic (Cell 17), not a trained/validated classifier.")


## Cell 22 — Download the Integration Model & Config (for `vanmitra_backend/` Satellite Agent)

**What it does:** Packages everything the FastAPI backend's `SatelliteAgent` (see
`SATELLITE_MODEL_INTEGRATION.md`, Sections 3 & 8) needs to run this notebook's trained model in
production, and downloads it as a single zip:

1. **`siamese_change_model.pt`** — the trained `SiameseChangeNet` weights (`model.state_dict()`),
   plus the architecture args (`in_channels`, `embed_dim`, `distance`) and preprocessing constant
   (`PATCH_SIZE`) needed to reconstruct the model with `model.load_state_dict(...)` inside
   `SatelliteAgent._run_analysis()`.
2. **`satellite_config.json`** — the exact thresholds this run used (`THETA`, `TAU`,
   `A_MIN_PIXELS`, resolution-feasibility cut-offs, domestic/farmland cut-off), in the same shape as
   `assets/ai_config/satellite_config.json` in the integration spec, so the backend's decision rules
   match this notebook's.
3. **`ozar_alerts.csv`** — the per-parcel results already written in Cell 21, included for reference.

**Output:** `vanmitra_module_b_integration_bundle.zip`, downloaded automatically in Colab (falls back
to a printed path if run outside Colab).

**Size/time:** instant — the model is tiny (well under the <30 MB on-device target noted in Cell 15).


In [ ]:
import shutil

# --- 1. Save the trained model (weights + everything needed to reload it) --------
model_checkpoint = {
    "state_dict": model.state_dict(),
    "architecture": {
        "in_channels": 2,
        "embed_dim": 32,
        "distance": model.distance,
    },
    "patch_size": PATCH_SIZE,
    "tau": TAU,
}
torch.save(model_checkpoint, "siamese_change_model.pt")
print("Saved: siamese_change_model.pt")

# --- 2. Save the config the backend's satellite_config.json should match ---------
satellite_config = {
    "ndvi_thresholds": {
        "theta_change_candidate": THETA,
        "siamese_decision_tau": TAU,
    },
    "cluster_thresholds": {
        "a_min_pixels": A_MIN_PIXELS,
        "pixel_area_sqm": PIXEL_AREA_SQM,
    },
    "resolution_feasibility_bands": {
        "reliable_min_sqm": RELIABLE_MIN_SQM,
        "marginal_min_sqm": MARGINAL_MIN_SQM,
    },
    "land_use_rule": {
        "domestic_area_threshold_sqm": DOMESTIC_AREA_THRESHOLD_SQM,
    },
    "area_discrepancy_tolerance_pct": AREA_DISCREPANCY_TOLERANCE_PCT,
    "model": {
        "file": "siamese_change_model.pt",
        "n_params": n_params,
        "patch_size": PATCH_SIZE,
    },
    "village_boundary_source": REAL_BOUNDARY_SOURCE,
    "earth_engine_imagery_used": EE_AVAILABLE,
}
with open("satellite_config.json", "w") as f:
    json.dump(satellite_config, f, indent=2)
print("Saved: satellite_config.json")

# --- 3. Bundle model + config + alerts CSV into one zip for download -------------
bundle_files = ["siamese_change_model.pt", "satellite_config.json"]
try:
    import os
    if os.path.exists("ozar_alerts.csv"):
        bundle_files.append("ozar_alerts.csv")
except Exception:
    pass

bundle_name = "vanmitra_module_b_integration_bundle"
with zipfile.ZipFile(f"{bundle_name}.zip", "w") as zf:
    for fname in bundle_files:
        zf.write(fname)
print(f"Bundled: {bundle_name}.zip  (", ", ".join(bundle_files), ")")

# --- 4. Trigger download (Colab) or point to the local path ----------------------
try:
    from google.colab import files
    files.download(f"{bundle_name}.zip")
    print("Download triggered — check your browser's downloads.")
except Exception:
    print(f"Not running in Colab — find the bundle at: {os.path.abspath(bundle_name + '.zip')}")

print()
print("Next step: copy siamese_change_model.pt + satellite_config.json into")
print("  vanmitra_backend/assets/ai_config/  and load them inside")
print("  SatelliteAgent._run_analysis()  (see SATELLITE_MODEL_INTEGRATION.md, Section 7).")
